## Prerequisistes

In [1]:
%load_ext autoreload
%autoreload 2

%pip install -qq --upgrade pip pandas
%pip install -qq numpy
%pip install -qq matplotlib
%pip install -qq seaborn
%pip install -qq jupyter
%pip install -qq qdrant-client
%pip install -qq sentence-transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np
from python import eda_utils as eda
from python import rag_utils as rag
import ipywidgets as widgets
from IPython.display import display

In [3]:
eda_selector = widgets.Dropdown(
    options=[True, False],
    value=False,
    description="EDA:"
)

display(eda_selector)

eda_enabled = eda_selector.value

Dropdown(description='EDA:', index=1, options=(True, False), value=False)

## Exploratory Data Analysis

In [4]:
if eda_enabled:
    df = eda.load_data('products.csv')
    df = eda.clean_column_names(df)
    df = eda.drop_empty_columns(df)
else:
    df = pd.read_csv('products_clean.csv')

In [5]:
if eda_enabled:
    df = eda.aggregate_by_handle(df, separator=" ; ")
    df = df.drop(columns=['Handle'])
    df = eda.compress_variations(df, column_name='Variations')

In [6]:
if eda_enabled:
    eda.display_low_cardinality_info(df, threshold=1)

In [7]:
if eda_enabled:
    df = eda.drop_low_cardinality_columns(df, threshold=1)

In [8]:
if eda_enabled:
    eda.display_low_cardinality_info(df, threshold=4)

In [9]:
if eda_enabled:
    df = df.drop(columns=['Published', 'Option1 Name', 'Option2 Name', 'Variant Taxable', 'Complementary Heading',  'Product rating count', 'Dietary preferences', 'Plant characteristics'])
    df = df.drop(columns=['Status', 'Image Src', 'Image Position', 'Variant SKU'])

In [10]:
if eda_enabled:
    eda.display_low_cardinality_info(df, threshold=4)

In [11]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.85)

In [12]:
if eda_enabled:
    df = df.drop(columns=['Cost per item', 'Shape', 'Variant Image'])

In [13]:
if eda_enabled:
    eda.analyze_column(df, 'Option2 Value', top_n=30)
    eda.analyze_column(df, 'Tags', top_n=30)
    eda.analyze_column(df, 'Option1 Value', top_n=30)
    eda.analyze_column(df, 'Variant Compare At Price', top_n=30)
    eda.analyze_column(df, 'Google Shopping / Google Product Category', top_n=30)
    eda.analyze_column(df, 'Badges', top_n=30)
    eda.analyze_column(df, 'Custom Label #1', top_n=30)
    eda.analyze_column(df, 'Custom Label #2', top_n=30)
    eda.analyze_column(df, 'Custom Label #3', top_n=30)
    eda.analyze_column(df, 'Custom Label #4', top_n=30)
    eda.analyze_column(df, 'Custom Label #5', top_n=30)
    eda.analyze_column(df, 'Floral Arrangements Type', top_n=30)
    eda.analyze_column(df, 'Variations', top_n=30)
    eda.analyze_column(df, 'Allergen information', top_n=30)

In [14]:
if eda_enabled:
    df = df.drop(columns=['Option2 Value', 'Variant Compare At Price', 'Google Shopping / Google Product Category', 'Custom Label #3'])

In [15]:
if eda_enabled:
    df['Option1 Value'] = df['Option1 Value'].replace('Default Title', np.nan)
    df = df.rename(columns={
        'Option1 Value': 'Details',
        'Badges': 'Tags2',
        'Custom Label #1': 'Sezon',
        'Custom Label #2': 'Parfum',
        'Custom Label #3': 'Polen',
        'Custom Label #4': 'Cadou',
        'Custom Label #5': 'Dimensiune'
    })
    df['Dimensiune'] = df['Dimensiune'].str.replace('Dimensiune ', '', regex=False).str.strip()

In [16]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.85)

In [17]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.5, max_threshold=0.85)

In [18]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.25, max_threshold=0.5)

In [19]:
if eda_enabled:
    df = eda.clean_html_column(df, column_name='Body (HTML)')

In [20]:
if eda_enabled:
    for col in df.columns:
        eda.analyze_column(df, col, top_n=30)

In [21]:
if eda_enabled:
    print(f"\n✨ FINAL DATASET: {df.shape[0]} Products, {df.shape[1]} Columns ✨")

    file_to_save = 'products_clean.csv'
    if os.path.exists(file_to_save):
        os.remove(file_to_save)
        print(f"Deleted old version of {file_to_save}")

    df.to_csv(file_to_save, index=False)
    print(f"Successfully saved new {file_to_save}")

In [22]:
if eda_enabled:
    docs_pentru_qdrant = eda.export_for_qdrant(df)

In [23]:
rag.build_and_index_rag(df, output_json='qdrant_payload.json', qdrant_url="http://localhost:40081")

📦 Construim Payload-ul și Vectorii...
💾 Backup salvat în qdrant_payload.json.
🧠 Încărcăm modelul de Embedding: paraphrase-multilingual-MiniLM-L12-v2...
🎯 Încărcăm modelul SOTA de Reranking: cross-encoder/nli-distilroberta-base...
🔌 Ne conectăm la Qdrant (http://localhost:40081)...
📁 Pregătim colecția 'city_flowers_catalog'...
⚙️ Generăm vectorii (Embedding)...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

🚀 Încărcăm în Qdrant...
✅ Baza de date a fost populată cu succes!


In [24]:
# ==========================================
# CELULA 2: TESTĂM REȚEAUA SOTA (Hibrid + Reranker)
# ==========================================
# Aici simulezi ce va extrage Mistral Router-ul:
simulated_state = {
  "filters": {
    "price_max": 400,
    "category": "Buchete Flori"
  },
  "semantic_query": "Vreau un buchet cu lalele și un aspect elegant, de primăvară"
}

# Aici testam functia SOTA care cauta si aplica cross-encoderul
rezultate_sota = rag.sota_hybrid_search(simulated_state, final_top_k=3, qdrant_url="http://localhost:40081")

# Printam contextul asa cum il va vedea modelul RoMistral final
print(rag.format_context_for_mistral(rezultate_sota))

🔌 Ne conectăm la Qdrant (http://localhost:40081)...
❌ Eroare la căutarea în Qdrant. Verifică versiunea librăriei 'qdrant-client'. Detalii: 'QdrantClient' object has no attribute 'qdrant'
Nu am găsit produse potrivite.
